In [ ]:
import os
import json
import pandas as pd

from scipy import stats
from sklearn.metrics import mean_absolute_error, median_absolute_error

# Overview

This is a notebook that provides some convenience scripts for gathering the predictions across various datasets. We can organize this notebook in two different sections. First, we will have code to combine all of the predictions (which will include cases and controls). Secondly, we will have code to filter by controls and non-controls.

## Global Definitions

In [ ]:
ANALYSIS_DIR = "" # Set this to the directory where you have saved your analysis results
DATA_DIR = "" # Set this to the directory where you have saved your data
AGING_CLOCKS = [
     "horvath2013", "hannum", "skinandblood", "altumage", "zhangen", "zhangblup", "corticalage",
     "grimage2", "grimage", "phenoage", "zhangmortality", "systemsage", "systemsageblood", "systemsagebrain",
     "pchorvath", "pchannum", "pcphenoage", "brainage"
]

## Functions

In [ ]:
def gather_predictions(analysis_dir=ANALYSIS_DIR):
    predictions = []
    for folder in os.listdir(analysis_dir):
        if not folder.startswith("GSE"):
            continue
        folder_path = os.path.join(analysis_dir, folder)
        prediction_path = os.path.join(folder_path, "predictions.csv")
        if not os.path.exists(prediction_path):
            print(f"Predictions not found for {folder}")
            continue
        prediction_df = pd.read_csv(prediction_path)
        prediction_df["Accession_Code"] = folder
        predictions.append(prediction_df)
    if not predictions:
        print("No predictions found.")
        return pd.DataFrame()  # Return empty DataFrame if no predictions found
    predictions_df = pd.concat(predictions, ignore_index=True)
    predictions_df = predictions_df.sort_values(by=["Accession_Code", "Sample"])
    return predictions_df

def gather_metadata(data_dir=DATA_DIR):
    metadata_dfs = []
    for folder in os.listdir(data_dir):
        if not folder.startswith("GSE"):
            continue
        folder_path = os.path.join(data_dir, folder)
        metadata_path = os.path.join(folder_path, "sample_metadata.csv")
        extraction_protocol_path = os.path.join(folder_path, "extraction_protocol.json")
        if not os.path.exists(extraction_protocol_path):
            print(f"Extraction protocol not found for {folder}")
            continue
        if not os.path.exists(metadata_path):
            print(f"Metadata not found for {folder}")
            continue
        extraction_protocol = json.load(open(extraction_protocol_path))
        metadata_df = pd.read_csv(metadata_path)
        if extraction_protocol["disease_status"].get("status", False) == "resolved":
            control_value = extraction_protocol["disease_status"]["extraction"]["control_value"]
            metadata_df["is_control"] = metadata_df["sample_id"].apply(lambda x: x == control_value)
            metadata_dfs.append(metadata_df)
        else:
            metadata_df["is_control"] = True
    if not metadata_dfs:
        print("No metadata found.")
        return pd.DataFrame()  # Return empty DataFrame if no metadata found
    metadata_df = pd.concat(metadata_dfs, ignore_index=True)
    metadata_df = metadata_df.sort_values(by=["Accession_Code", "Sample"])

def combine_predictions_and_metadata(predictions_df, metadata_df):
    merged_df = pd.merge(predictions_df, metadata_df[["Accession_Code", "Sample", "is_control"]], left_on=["Accession_Code", "Sample"], right_on=["Accession_Code", "Sample"], how="inner")
    merged_df.drop(columns=["Sample"], inplace=True)
    return merged_df

def construct_summarized_prediction_df(df, clocks=AGING_CLOCKS):
    rows = []
    accession_codes = sorted(df["Accession_Code"].unique())
    for accession_code in accession_codes:
        for clock in clocks:
            if clock not in df.columns:
                continue
            clock_subset = df[df["Accession_Code"] == accession_code].dropna(subset=[clock.lower(), "age"])
            if len(clock_subset) < 2:
                continue
        rows.append(
            {
                "Accession_Code": accession_code,
                "Clock": clock,
                "MAE": mean_absolute_error(clock_subset["age"], clock_subset[clock.lower()]),
                "MedAE": median_absolute_error(clock_subset["age"], clock_subset[clock.lower()]),
                "Pearson_R": stats.pearsonr(clock_subset["age"], clock_subset[clock.lower()])[0],
            }
        )
    summarized_df = pd.DataFrame(rows)
    summarized_df = summarized_df.sort_values(by=["Accession_Code", "MAE"])
    return summarized_df

## Retrieve Data

In [ ]:
predictions_df = gather_predictions()
metadata_df = gather_metadata()
combined_df = combine_predictions_and_metadata(predictions_df, metadata_df)

## All Predictions

This section gathers all of the available prediction files into a single dataframe. You can save from here, or inspect it interactively.

In [ ]:
all_predictions_df = combined_df.copy()
all_predictions_summarized = construct_summarized_prediction_df(all_predictions_df)
display(all_predictions_summarized)

## Filtered Predictions

For our paper, we looked specifically at the control samples across datasets. We show how you can reproduce this in the following cells.

In [ ]:
control_predictions_df = combined_df[combined_df["is_control"]].copy()
control_predictions_summarized = construct_summarized_prediction_df(control_predictions_df)
display(control_predictions_summarized)